# Dense image-language policy

## 1. Load shared code
This notebook adds the instruction to the images. The word embedding and GRU learn from scratch using the two training-task instructions. This is not a pretrained language model and does not establish general language understanding.

In [1]:
from pathlib import Path
import sys, json
import torch, torchvision
project_dir = Path.home() / "t2-project"
sys.path.insert(0, str(project_dir))
from t2.train import train, load_policy
from t2.data import preparation, vocabulary, Demonstrations
manifest, normalization, manifest_hash = preparation(project_dir)
print("Project:", project_dir)

Project: /home/njmcmil/t2-project


## 2. Train the dense baseline
The encoder supplies all 256 spatial tokens per camera. Coordinate and camera embeddings identify where features came from. A language query attends over tokens before the shared action head predicts seven normalized actions. Start with the same 10-epoch pilot budget; agree any final budget with Ashish using validation only.

In [2]:
dense_run = train(project_dir, "dense", seed=0, epochs=10)
print(dense_run)

Run: /home/njmcmil/t2-project/runs/dense_seed0_20261117T211545_858127Z
Epoch 1/10: train=0.97510, validation=1.00946
Epoch 2/10: train=0.92842, validation=0.89419
Epoch 3/10: train=0.81998, validation=0.83333
Epoch 4/10: train=0.73880, validation=0.73448
Epoch 5/10: train=0.69534, validation=0.73535
Epoch 6/10: train=0.65652, validation=0.69298
Epoch 7/10: train=0.60929, validation=0.67238
Epoch 8/10: train=0.58933, validation=0.62903
Epoch 9/10: train=0.56244, validation=0.61968
Epoch 10/10: train=0.54447, validation=0.62178
/home/njmcmil/t2-project/runs/dense_seed0_20261117T211545_858127Z


## 3. Read validation results
The best checkpoint was selected by validation loss and reloaded automatically. Low loss is not simulator success.

In [3]:
print((dense_run / "validation_summary.json").read_text())

{
  "selected_epoch": 9,
  "validation_mse": 0.619675495972236,
  "per_task": {
    "put_the_bowl_on_the_plate": 0.697488769836409,
    "put_the_bowl_on_the_stove": 0.5439941026334179
  },
  "checkpoint_reload_verified": true
}


## 4. Try both development tasks
This uses training scenes, not held-out states. Saved videos label tasks, seeds, and simulator success. A new run directory avoids overwriting prior attempts.

In [4]:
from datetime import datetime, timezone
from t2.simulation import development_case, rollout
development_dir = dense_run / "development" / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
for task in manifest["tasks"]:
    result = rollout(project_dir, dense_run / "best_model.pt", development_case(project_dir, task), development_dir)
    print(result["task"], result["success"], result["steps"])

[robosuite WARNING] No private macro file found! (__init__.py:7)
[robosuite WARNING] It is recommended to use a private macro file (__init__.py:8)
[robosuite WARNING] To setup, run: python /home/njmcmil/t2-project/.venv/lib/python3.10/site-packages/robosuite/scripts/setup_macros.py (__init__.py:9)
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
put_the_bowl_on_the_plate put_the_bowl_on_the_plate_demo_5 success= False
put_the_bowl_on_the_plate False 300
[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
put_the_bowl_on_the_stove put_the_bowl_on_the_stove_demo_34 success= False
put_the_bowl_on_the_stove False 300


## 5. Second training seed
The proposal calls for seeds 0 and 1. Keep the final data and update budgets equal across comparison variants. This cell trains another model; it does not resume seed 0.

In [5]:
dense_seed1_run = train(project_dir, "dense", seed=1, epochs=10)

Run: /home/njmcmil/t2-project/runs/dense_seed1_20261117T211813_215453Z
Epoch 1/10: train=0.97423, validation=1.00354
Epoch 2/10: train=0.97255, validation=0.99708
Epoch 3/10: train=0.92539, validation=0.90231
Epoch 4/10: train=0.85839, validation=0.86360
Epoch 5/10: train=0.77270, validation=0.80807
Epoch 6/10: train=0.71336, validation=0.71901
Epoch 7/10: train=0.66935, validation=0.69063
Epoch 8/10: train=0.63740, validation=0.68854
Epoch 9/10: train=0.61238, validation=0.66332
Epoch 10/10: train=0.58764, validation=0.64212
